# Notebook 07 — Check inference paths and prepare balanced elementary data

The lower-learning-rate trial completed, but did not improve the 50-question validation score. This notebook prepares the next controlled experiment in two parts:

1. **CPU:** create 1,680 checked elementary examples and 120 held-out-template development diagnostics. Replace 1,680 original GSM8K training rows while preserving the total 6,726 rows and all original validation/test files and three demonstrations.
2. **GPU:** compare the same 50 validation prompts through the direct base, repeated direct base, zero-initialized LoRA, and disabled-adapter paths. This investigates path-dependent generation with **200 predictions and zero optimizer updates**.

**Kaggle:** enable Internet and a T4 GPU. Attach only the **original saved Notebook 01 output**. Earlier adapters and notebook outputs are not required. One GPU is used.

Allow roughly **30–50 minutes**, depending on setup and generation lengths. This is an estimate. This notebook does not train another adapter; review the diagnostic and data sample before the matched control/data-mixture experiment.

The 120 synthetic diagnostic questions use held-out wording and per-skill number tuples. They are development checks within these six simple skill families, not evidence of broad maths ability.


## 1. Fetch the shared GitHub code


In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys
import tempfile

REPOSITORY = "https://github.com/sahan24854/Fine-Tuning-LLM-for-math.git"
PROJECT_REF = os.environ.get("MATH_PROJECT_REF", "main")
if PROJECT_REF.startswith("-"):
    raise ValueError("Use a branch, tag, or full commit SHA as MATH_PROJECT_REF")
PROJECT = Path(tempfile.mkdtemp(prefix="math-project-"))
subprocess.run(["git", "init", "--quiet", str(PROJECT)], check=True)
subprocess.run(["git", "-C", str(PROJECT), "remote", "add", "origin", REPOSITORY], check=True)
subprocess.run(["git", "-C", str(PROJECT), "fetch", "--depth", "1", "origin", PROJECT_REF], check=True)
subprocess.run(["git", "-C", str(PROJECT), "checkout", "--detach", "--quiet", "FETCH_HEAD"], check=True)
CODE_COMMIT = subprocess.check_output(["git", "-C", str(PROJECT), "rev-parse", "HEAD"], text=True).strip()
print("Repository:", REPOSITORY)
print("Code commit:", CODE_COMMIT)
print("Project checkout:", PROJECT)


## 2. Verify the original bundle and choose a fresh output directory

Keep the original split, pinned model/tokenizer, demonstrations and scorer. Folder/ZIP discovery is supported. If more than one bundle is attached, set `MATH_BUNDLE_PATH` to the original Notebook 01 bundle.


In [ ]:
import hashlib
import shutil
from datetime import datetime, timezone

INPUT_ROOT = Path(os.environ.get("MATH_BUNDLE_PATH", "/kaggle/input"))
BUNDLE = Path(subprocess.check_output([sys.executable, str(PROJECT / "scripts/find_bundle.py"), str(INPUT_ROOT)], text=True).strip())
_ = subprocess.run([sys.executable, str(PROJECT / "scripts/verify_bundle.py"), str(BUNDLE)], check=True)
BUNDLE_HASH = hashlib.sha256((BUNDLE / "manifest.json").read_bytes()).hexdigest()
STAMP = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
RUN_ROOT = Path("/kaggle/working") / f"balanced-preparation-{BUNDLE_HASH[:8]}-{CODE_COMMIT[:8]}-{STAMP}"
DATA_OUTPUT, PATH_OUTPUT = RUN_ROOT / "data", RUN_ROOT / "inference_paths"
print("Original bundle:", BUNDLE)
print("Output:", RUN_ROOT)
print("Data settings:", (PROJECT / "configs/balanced_data.json").read_text())


## 3. Prepare the isolated environment

The pinned training profile supplies Transformers, PyTorch and PEFT for the zero-adapter diagnostic. It does not replace Kaggle kernel packages. Missing-`wrapt` startup warnings are separate from an actual command failure.


In [ ]:
ENVIRONMENT = Path(tempfile.mkdtemp(prefix="math-preparation-env-"))
_ = subprocess.run([sys.executable, str(PROJECT / "scripts/setup_environment.py"),
    "--environment", str(ENVIRONMENT), "--profile", "training"], check=True)
PYTHON = ENVIRONMENT / "bin" / "python"
RUN_ENV = {**os.environ, "CUDA_VISIBLE_DEVICES": "0"}


## 4. Run the local correctness checks

These tests check every generated example, balanced counts, independent arithmetic results, held-out templates/tuples, overlap filtering, unchanged original evaluation files and demonstrations, refusal of truncation, and the real tiny-model inference paths. No remote model weights are needed for these tests.


In [ ]:
for test_file in ("test_balanced_data.py", "test_inference_paths.py"):
    _ = subprocess.run([str(PYTHON), "-m", "unittest", "discover", "-s", "tests", "-p", test_file, "-b", "-v"],
        cwd=PROJECT, check=True, env=RUN_ENV)


## 5. Prepare and inspect the data variant

Train on 5,046 original GSM8K rows plus 1,680 synthetic rows, 280 in each skill: addition, subtraction, equal groups, exact division, multiplication then addition, and multiplication then subtraction. Retain all three original demonstrations. The 25% replacement is a declared first hypothesis, not an optimized ratio.

Solutions are short, machine-checked integer equations. This changes both skill coverage and target-solution style. The same row count preserves the planned optimizer-update count under the old schedule, but shorter responses change supervised-token exposure. The summary reports both token totals; do not claim an identical token budget.

The original validation/test files, tokenizer and demonstrations stay fixed. The new bundle gets a different identity. Keep it separate from Notebook 01; old comparison scripts intentionally reject changed bundle identities.


In [ ]:
_ = subprocess.run([str(PYTHON), str(PROJECT / "scripts/prepare_balanced_data.py"),
    "--bundle", str(BUNDLE), "--output", str(DATA_OUTPUT)], check=True, cwd=PROJECT, env=RUN_ENV)
DATA_SUMMARY = json.loads((DATA_OUTPUT / "summary.json").read_text())
print(json.dumps(DATA_SUMMARY, indent=2))
print("Training sample: one example of each skill/wording template")
for line in (DATA_OUTPUT / "review_sample.jsonl").read_text().split("\n"):
    if line:
        row = json.loads(line)
        print("\n", row["template_id"], row["question"])
        print(row["solution"], "Final answer:", row["answer"], sep="\n")


## 6. Check generation through four inference paths

Use identical prompts, order, batches of four, greedy settings, 512-token cap, stopping, tokenizer and loaded base weights. First repeat the direct-base path. Then attach an untrained zero-initialized LoRA adapter, and finally disable that adapter. No parameter update takes place.

The probe saves all raw completions and identifies differences in text, stop reason or generated length. A mismatch can expose a path dependence; this probe alone cannot prove its numerical cause or reproduce every historical Trainer preparation step. Even matching accuracy does not mean matching answers.


In [ ]:
if not shutil.which("nvidia-smi"):
    raise RuntimeError("Enable a T4 GPU in Kaggle settings for this diagnostic.")
_ = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader"], check=True)
_ = subprocess.run([str(PYTHON), str(PROJECT / "scripts/check_inference_paths.py"),
    "--bundle", str(BUNDLE), "--output", str(PATH_OUTPUT)], check=True, cwd=PROJECT, env=RUN_ENV)
PATH_SUMMARY = json.loads((PATH_OUTPUT / "summary.json").read_text())
print(json.dumps(PATH_SUMMARY, indent=2))
if not PATH_SUMMARY["reproducibility_gate_passed"]:
    print("REVIEW REQUIRED: inference paths differ. Preserve the outputs and investigate before training.")


## 7. Verify and save the complete output

Retain both `data/` and `inference_paths/`, including their manifests. Share `inference_paths/summary.json` and `data/summary.json`, or push the executed notebook. If paths differ, also share the four prediction files. Training is a separate next step after review.

The next candidate/control comparison should use the same inference path and fresh adapters, fixed `2e-4` learning rate, three epochs, effective batch 32 and loss-based checkpoint selection. Returning to `2e-4` is an explicit reference configuration, not proof that it is optimal. The old adapters remain historical evidence; they do not replace a matched control for strong causal claims.


In [ ]:
for output in (DATA_OUTPUT, PATH_OUTPUT):
    manifest = json.loads((output / "result_manifest.json").read_text())
    for name, expected in manifest["files_sha256"].items():
        if hashlib.sha256((output / name).read_bytes()).hexdigest() != expected:
            raise RuntimeError(f"Checksum mismatch: {output / name}")
ARCHIVE = Path(shutil.make_archive(str(RUN_ROOT), "zip", root_dir=RUN_ROOT))
print("Balanced data bundle:", DATA_OUTPUT / "bundle")
print("Data summary:", DATA_OUTPUT / "summary.json")
print("Inference diagnostic summary:", PATH_OUTPUT / "summary.json")
print("Complete ZIP:", ARCHIVE)
print("Save Version with outputs. Review before training the data-mixture candidate.")
